# Week 5 — Regression and factor models

> Part of the open-source teaching project **quant-math-roadmap**.
> For **education and research methodology only** — not investment advice; no result here represents a profitable or investable strategy.

## Learning objectives

- Derive and implement OLS in matrix form.
- Estimate a CAPM-style market beta and interpret it.
- Fit a multi-factor model and inspect the residuals.
- Compute rolling betas and understand omitted variable bias.

## Estimated study time

About 9–11 hours.

## Prerequisites

- Matrix operations from Week 1
- Standard errors from Week 4

## External resources

- [NTU OpenCourseWare Statistics I and Introductory Econometrics](https://ocw.aca.ntu.edu.tw/courses/112S103)
- [MIT OpenCourseWare 18.06SC Linear Algebra](https://ocw.mit.edu/courses/18-06sc-linear-algebra-fall-2011/)

> External resources are linked for reference only; this project does not reproduce any copyrighted course material.

In [ ]:
# Teaching style setup (deterministic look, consistent figures)
import matplotlib as _mpl
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## Concepts

### OLS in matrix form

For the model $y = X\beta + \varepsilon$, the least-squares solution is

$$ \hat\beta = (X^\top X)^{-1} X^\top y. $$

Geometrically, $X\hat\beta$ is the **projection** of $y$ onto the subspace spanned by the columns of $X$; the residual $y - X\hat\beta$ is orthogonal to that subspace.

### Financial meaning

In the CAPM-style regression $r_{\text{asset}} = \alpha + \beta\, r_{\text{market}} + \varepsilon$, $\beta$ measures the asset's exposure to the market. **But remember: a regression coefficient does not automatically become a tradable signal** — it merely describes a historical co-movement.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from quant_math_roadmap.math.statistics import ols_fit
from quant_math_roadmap.math.linear_algebra import add_intercept, ols_beta

rng = np.random.default_rng(2024)
n = 600
market = rng.normal(0.0003, 0.011, n)
true_beta, true_alpha = 1.2, 0.0001
idiosyncratic = rng.normal(0.0, 0.008, n)
asset = true_alpha + true_beta * market + idiosyncratic
print('Generated synthetic market and asset returns, n =', n)

### Hand-rolled OLS vs statsmodels

In [ ]:
fit = ols_fit(market, asset, add_const=True, feature_names=['market'])
print(fit.summary())
print()
sm_fit = sm.OLS(asset, sm.add_constant(market)).fit()
print('statsmodels coefficients:', np.round(sm_fit.params, 6))
print('our coefficients        :', np.round(fit.params, 6))
assert np.allclose(fit.params, sm_fit.params)

The estimated beta should be close to the true value 1.2. Our hand-rolled OLS matches `statsmodels` exactly, confirming $\hat\beta = (X^\top X)^{-1}X^\top y$.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(market, asset, s=8, alpha=0.4, label='Observations')
grid = np.linspace(market.min(), market.max(), 100)
ax.plot(grid, fit.params[0] + fit.params[1] * grid,
        label=f'OLS fitted line (beta={fit.params[1]:.3f})')
ax.set_title('CAPM-style regression: asset return vs market return')
ax.set_xlabel('Market return')
ax.set_ylabel('Asset return')
ax.legend()
plt.show()

### Multi-factor model

In [ ]:
value_factor = rng.normal(0.0, 0.007, n)
size_factor = rng.normal(0.0, 0.006, n)
asset_multi = (0.0001 + 1.1 * market + 0.6 * value_factor
               - 0.3 * size_factor + rng.normal(0, 0.005, n))
X = np.column_stack([market, value_factor, size_factor])
multi_fit = ols_fit(X, asset_multi, add_const=True,
                    feature_names=['market', 'value', 'size'])
print(multi_fit.summary())

Each coefficient is the exposure to that factor 'holding the other factors fixed'. $R^2$ measures how much of the return variance the model explains — but a high $R^2$ does **not** mean profitability.

### Rolling beta

In [ ]:
asset_s = pd.Series(asset)
market_s = pd.Series(market)
window = 120
rolling_beta = []
for end in range(window, n + 1):
    sl = slice(end - window, end)
    b = ols_beta(add_intercept(market_s.iloc[sl].to_numpy()),
                 asset_s.iloc[sl].to_numpy())
    rolling_beta.append(b[1])
rolling_beta = pd.Series(rolling_beta, index=range(window, n + 1))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(rolling_beta.index, rolling_beta.values, label=f'{window}-period rolling beta')
ax.axhline(true_beta, linestyle='--', label=f'True beta = {true_beta}')
ax.set_title('Rolling beta estimates over time')
ax.set_xlabel('Window end position')
ax.set_ylabel('Estimated beta')
ax.legend()
plt.show()

Even though the true beta is constant, the rolling estimate still oscillates around it — that is the sampling uncertainty of estimation. With real data, betas also **genuinely change over time**.

### Heteroskedasticity and robust standard errors (HC0 / HC1)

Financial data often exhibit **heteroskedasticity**: the error variance is not constant (for example, it grows with market volatility). In that case the OLS **coefficient estimates remain unbiased**, but the classical standard errors are wrong — significance tests get misled. White's (1980) **sandwich estimator** re-estimates the coefficient covariance using only 'the squared actual residuals', with no assumption about the error structure:

$$ \widehat{\mathrm{Var}}(\hat\beta)_{HC0} = (X^\top X)^{-1} X^\top \mathrm{diag}(e_i^2)\, X (X^\top X)^{-1} $$

Below we deliberately build a dataset whose error variance grows with $|x|$ and compare classical against robust standard errors.

In [ ]:
# Error std = 0.5 + |x| -> textbook-grade heteroskedasticity
x_het = rng.standard_normal(800)
y_het = 1.0 + 2.0 * x_het + rng.standard_normal(800) * (0.5 + np.abs(x_het))

classic = ols_fit(x_het, y_het, feature_names=['x'])
robust = ols_fit(x_het, y_het, feature_names=['x'], robust='HC1')
print('Coefficients identical:', np.allclose(classic.params, robust.params))
print(f'Classical std error of the slope  = {classic.std_errors[1]:.4f}')
print(f'HC1 robust std error of the slope = {robust.std_errors[1]:.4f}')
print('Under heteroskedasticity, classical standard errors clearly understate uncertainty -> t statistics are inflated.')

**Takeaway**: when reporting financial regressions, defaulting to robust standard errors is good practice. Note that they only fix the inference (standard errors, t, p-values), not the coefficients themselves — the model's explanatory power is unchanged; what changes is your confidence in the significance.

### Residual inspection and omitted variable bias

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(range(len(multi_fit.residuals)), multi_fit.residuals, s=8, alpha=0.4)
ax.axhline(0.0, linestyle='--')
ax.set_title('Residuals of the multi-factor model')
ax.set_xlabel('Observation index')
ax.set_ylabel('Residual')
plt.show()

In [ ]:
# Deliberately omit the value factor and watch the beta get distorted
biased = ols_fit(market, asset_multi, add_const=True, feature_names=['market'])
full = ols_fit(X, asset_multi, add_const=True,
               feature_names=['market', 'value', 'size'])
print('market coefficient with omitted variable:', round(biased.params[1], 4))
print('market coefficient in the full model    :', round(full.params[1], 4))
print('If an omitted variable correlates with an included one, the estimated coefficient is biased.')

## Exercises

Work through these in order. **Basic exercises** consolidate the definitions, **applied exercises** are hands-on coding, and the **reflection question** connects the mathematics to backtesting and research methodology.

> The main notebook ships runnable starter code for each coding exercise. Full reference answers live in the matching `_solution` notebook under `notebooks/en/solutions/`.

### Basic exercises

1. Explain what OLS does in the language of geometric projection.
2. Explain what the intercept, beta, residuals and $R^2$ each represent.
3. Why does 'the regression coefficient is significant' not mean 'we can trade on it'?

### Applied exercises

In [ ]:
# Applied exercise 1: without using ols_fit, estimate beta directly from the matrix formula (X^T X)^-1 X^T y
# and compare against ols_fit (remember to add the intercept column).
Xc = add_intercept(market)
my_beta = None  # TODO: np.linalg.solve(Xc.T @ Xc, Xc.T @ asset)
if my_beta is not None:
    print('hand-computed beta:', np.round(my_beta, 6))

In [ ]:
# Applied exercise 2: change the rolling window to 60 periods and observe how the volatility of the rolling beta changes.
win = 60
betas = []  # TODO: compute the rolling beta with win, mirroring the code above
print('Once done, compare the volatility of the 60-period and 120-period estimates.')

### Reflection question

1. Suppose a regression tells you some factor is 'significant' for next-period returns. Before turning it into a backtest signal, what does Week 4 (multiple testing) warn you about, and what does Week 8 (leakage) warn you about?

## Quiz (self-check)
Answer the multiple-choice questions, then run the next cell to check yourself. Answers are stored as hashes, not plaintext.

**Q1. What is the matrix solution β̂ of OLS?**
- A. (XᵀX)⁻¹Xᵀy
- B. Xᵀy
- C. X⁻¹y
- D. (XXᵀ)⁻¹yX

**Q2. What does R² measure?**
- A. How profitable the strategy is
- B. The proportion of variance in the dependent variable explained by the model
- C. The magnitude of the coefficients
- D. The sum of the residuals

**Q3. Heteroskedasticity mainly affects which part of OLS?**
- A. The coefficient estimates
- B. The standard errors (and thus the t statistics)
- C. R²
- D. The intercept

**Q4. What do HC0/HC1 robust standard errors change?**
- A. The regression coefficients
- B. The standard errors
- C. The residuals
- D. R²

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: fill in 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: '881e66b9d3c17d9a', 2: '07f7b302f3c9c15d', 3: '2e30a5ff5a744c2f', 4: 'b22e68d5e3df82b5'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: unanswered')
        continue
    _h = _hashlib.sha256(f'qmr-w5-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ correct' if _ok else '✘ incorrect'))
print(f'Score: {_n_correct} / {len(my_answers)}')

## Common mistakes

- **Forgetting to add an intercept column to the design matrix.**
- **Treating a high $R^2$ as evidence that the strategy is profitable.**
- **Ignoring that heteroskedasticity invalidates plain OLS standard errors.**
- **Omitting an important variable and incurring omitted variable bias.**
- **Treating a regression coefficient directly as a tradable signal.**

## After this week, you should be able to

- [ ] Derive and implement $\hat\beta=(X^\top X)^{-1}X^\top y$.
- [ ] Interpret the intercept, beta, residuals and $R^2$.
- [ ] Compute rolling betas and explain their volatility.
- [ ] Explain omitted variable bias.

## References and attribution

- Every explanation, example and exercise in this notebook is **original** to this project.
- Recommended external resources: [`docs/resources.md`](../../docs/resources.md).
- Concept notes: [`docs/math/`](../../docs/math/) and [`docs/finance/`](../../docs/finance/).

### Privacy and disclaimer

- This notebook contains no real personal information.
- This notebook uses only reproducible synthetic data and needs no network access.
- This notebook makes no claim of real-world trading profitability.